# Phase 11 — Evaluation

Runs the query set through the **full system and its baselines**, and reports:

1. **Retrieval quality** — Precision@k, nDCG, MRR, Hit@k for the full system vs
   a keyword-only baseline and the stage ablations.
2. **Grounding** — do the answers cite passages that were *actually retrieved*?
   (deterministic citation verification — free, no judge model needed)
3. **Refusal behaviour** — honest refusal on the deliberately-unanswerable
   queries, and **over-refusal** on answerable ones.
4. **Ungrounded control** — the same LLM with no corpus, to show what grounding buys.
5. **RAGAS faithfulness / answer-relevancy** — judged by the **local Llama**, so
   there is no API cost.

> **These numbers are PRELIMINARY.** The query set is *silver* (project-authored).
> Labels are book-level, so **Recall@k is reported as `n/a`** — it is undefined
> until passage-level labels exist. When the expert gold-set arrives, drop it in
> and re-run: Recall switches on automatically and nothing else changes.

**Requirements:** T4 GPU runtime + HF token (private corpus + Llama).


## Cell 1 — clone + dependencies

In [ ]:
import os, shutil, subprocess, sys

REPO_PATH = "/content/iks-rag-thesis"
REPO_URL = "https://github.com/ankit8453/iks-rag-thesis.git"

os.chdir("/content")
shutil.rmtree(REPO_PATH, ignore_errors=True)
env = os.environ.copy()
env["GIT_LFS_SKIP_SMUDGE"] = "1"          # dodge the LFS bandwidth stall
r = subprocess.run(["git", "clone", REPO_URL, REPO_PATH], env=env,
                   capture_output=True, text=True)
if r.returncode != 0:
    print(r.stdout); print(r.stderr)
    raise RuntimeError(f"git clone failed (exit {r.returncode})")
os.chdir(REPO_PATH); sys.path.insert(0, REPO_PATH)
print("Repo at:", os.getcwd())

# ---------------------------------------------------------------------------
# numpy MUST be pinned to 1.26.4 FIRST and alone. Colab's Python 3.13 ships
# numpy 2.x, but the pinned RAGAS/langchain 0.2 stack (and some wheels of
# datasets/sentence-transformers) were built against numpy 1.x. A mismatch
# throws "numpy.dtype size changed, may indicate binary incompatibility
# (Expected 96 ... got 88)" the moment numpy's C extensions load. Installing
# numpy 1.26.4 before everything else makes the whole stack agree.
# ---------------------------------------------------------------------------
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy==1.26.4", "scipy==1.13.1"])

DEPS = [
    "numpy==1.26.4", "scipy==1.13.1",     # pin together; both must match numpy 1.x
    "transformers>=4.44,<4.50", "accelerate>=0.34", "bitsandbytes>=0.44",
    "sentence-transformers>=3.0", "chromadb>=0.5", "rank_bm25>=0.2.2",
    "datasets>=2.20", "huggingface_hub>=0.24", "pyyaml>=6.0", "pydantic>=2.7",
]
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *DEPS],
                   capture_output=True, text=True)
if r.returncode != 0:
    print("
".join(r.stdout.splitlines()[-30:]))
    raise RuntimeError("pip install failed")

# RAGAS (contribution C3). It must be a stack that is INTERNALLY consistent:
# Colab ships langchain 0.3, but ragas 0.1.x expects the 0.2 layout, so an
# unpinned install fails with "No module named
# 'langchain_community.chat_models.vertexai'". Pin a mutually-compatible set.
# RAGAS stays optional — if this fails the harness still reports every other
# metric and records that RAGAS was skipped.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "ragas==0.1.21", "langchain==0.2.16", "langchain-community==0.2.17",
                "langchain-openai==0.1.25", "langchain-core<0.3",
                "numpy==1.26.4", "scipy==1.13.1"])   # re-assert last: ragas deps can drag 2.x

# Verify the pin actually took; if Colab had already imported a different numpy
# in this session, a runtime restart is required for it to take effect.
import importlib, numpy, scipy
importlib.reload(numpy)
print("numpy:", numpy.__version__, "| scipy:", scipy.__version__)
if not numpy.__version__.startswith("1.26") or not scipy.__version__.startswith("1.13"):
    print("
*** numpy/scipy not the pinned 1.26/1.13 in memory. "
          "Runtime > Restart session, then run this cell again "
          "(fast — deps are cached). ***")

print("deps installed")

from huggingface_hub import login
login()

import torch
print("cuda:", torch.cuda.is_available())


## Cell 2 — build the corpus collection (CPU) and score retrieval

Retrieval needs no generation, so this half runs before the LLM is loaded.

In [ ]:
from src.rag.corpus_loader import load_chunks_from_hf, build_chroma
from src.eval.query_set import load_query_set, answerable_cases
from src.eval.harness import run_full_evaluation

chunks = load_chunks_from_hf()
print("corpus chunks:", len(chunks))

collection = build_chroma(chunks, persist_dir="/content/eval_vecdb")

# build_chroma embeds on the GPU and leaves the model there. On a T4 that
# stolen VRAM is what later forces Llama to offload onto CPU RAM and crash the
# session, so reclaim it before anything else loads.
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# One CPU embedder + reranker, shared by every retrieval variant: retrieval is
# cheap on CPU, and the GPU must stay free for the LLM in Cell 3.
from sentence_transformers import SentenceTransformer
from sentence_transformers.cross_encoder import CrossEncoder
cpu_embedder = SentenceTransformer("BAAI/bge-large-en-v1.5", device="cpu")
cpu_reranker = CrossEncoder("BAAI/bge-reranker-base", device="cpu")
SHARED = {"embedder": cpu_embedder, "reranker": cpu_reranker}

cases = load_query_set()
print(f"queries: {len(cases)} "
      f"({len(answerable_cases(cases))} answerable, "
      f"{len(cases) - len(answerable_cases(cases))} deliberate negatives)")

K = 5
retrieval_only = run_full_evaluation(cases, collection=collection, k=K,
                                     shared_models=SHARED)
print()
print(retrieval_only["retrieval_table"])

### Reading the retrieval table

`full` is the system. **`keyword_only` is the baseline that matters** — if the
full system does not clearly beat it, the semantic bridge is not earning its
place. `dense_only` and `hybrid_no_rerank` isolate which stage contributes what.

`R@5` shows `n/a` by design: recall is undefined under book-level labels.

## Cell 3 — load the LLM and run generation + the ungrounded control

Deliberately **not** `load_all()`: that also loads the disease, soil and YOLO
models, which evaluation never touches. On a T4 they crowd the GPU, Llama then
offloads onto CPU RAM, and the session dies. Here only the LLM goes to the GPU;
retrieval keeps using the CPU embedder/reranker from Cell 2.

If this still runs out of memory, set `LLM_NAME` to
`"meta-llama/Llama-3.2-3B-Instruct"` — a smaller generator is a fair evaluation
as long as the same model is used for the grounded system and the ungrounded
control (it is: both use `generator` below). Record which one you used.

In [ ]:
from src.rag.generator import GroundedGenerator
from src.rag.pipeline import RAGPipeline
from src.rag.retriever import HybridRetriever

LLM_NAME = "meta-llama/Llama-3.1-8B-Instruct"   # -> Llama-3.2-3B-Instruct if OOM

# Retrieval stays on CPU; the GPU is reserved entirely for the generator.
retriever = HybridRetriever(collection, embedder=cpu_embedder, reranker=cpu_reranker)
generator = GroundedGenerator(model_name=LLM_NAME)      # 4-bit
pipeline = RAGPipeline(collection=collection, retriever=retriever,
                       generator=generator, default_k=K)
print("LLM ready:", LLM_NAME)

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM free {free/1024**3:.2f} / {total/1024**3:.2f} GiB")

full_eval = run_full_evaluation(
    cases, collection=collection,
    pipeline=pipeline,                # grounded system
    llm=generator,                    # same model, ungrounded, as the control
    k=K, shared_models=SHARED,
)

g = full_eval["generation"]
print(f"answerable queries      : {g.n_answerable}")
print(f"grounded answer rate    : {g.grounded_answer_rate:.2%}   "
      f"(answers citing a genuinely retrieved passage)")
print(f"valid citation rate     : {g.valid_citation_rate:.2%}   "
      f"(citations pointing at real retrieved passages)")
print(f"honest refusal rate     : {g.honest_refusal_rate:.2%}   "
      f"(on the {g.n_negative} unanswerable queries - higher is better)")
print(f"over-refusal rate       : {g.over_refusal_rate:.2%}   "
      f"(refusing answerable queries - LOWER is better)")

u = full_eval["ungrounded"]
print(f"\nungrounded control      : {u['n']} answers with no corpus at all; "
      f"unfounded citation rate {u['unfounded_citation_rate']:.2%}")

## Cell 4 — RAGAS faithfulness (judged by the local Llama, no API cost)

In [ ]:
from src.eval.config import EvalConfig, RAGASConfig
from src.eval.ragas_eval import RAGEvalSample, run_ragas_evaluation

# Build one RAGAS row per answerable query from the grounded system's output.
samples = []
for case in answerable_cases(cases):
    res = pipeline.answer(case.query, k=K)
    samples.append(RAGEvalSample(
        query=case.query,
        answer=getattr(res, "answer", "") or "",
        contexts=[c.text for c in getattr(res, "retrieved", [])],
        ground_truth=None,          # no expert reference answers yet
    ))
print("RAGAS samples:", len(samples))

# The config is frozen (immutable), so set the metric list at CONSTRUCTION time
# rather than assigning to cfg.ragas.metrics afterwards.
cfg = EvalConfig(ragas=RAGASConfig(metrics=["faithfulness", "answer_relevancy"]))

# ---------------------------------------------------------------------------
# Judge selection. Leave False for the free run; flip to True for a second pass
# with an INDEPENDENT judge (see the note below for why that is worth doing).
# ---------------------------------------------------------------------------
USE_PAID_JUDGE = False

judge = judge_emb = None
if USE_PAID_JUDGE:
    import getpass, os, subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "langchain-openai"])
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")  # never hard-code
    from langchain_openai import ChatOpenAI
    from langchain_community.embeddings import HuggingFaceEmbeddings
    judge = ChatOpenAI(model="gpt-4o-mini", temperature=0)   # cheap, independent
    judge_emb = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")  # local, free

scores = run_ragas_evaluation(
    samples, cfg,
    output_path="results/phase11_ragas_per_sample.csv",
    judge_llm=judge, judge_embeddings=judge_emb,
)
print("judge           :", "gpt-4o-mini (independent)" if USE_PAID_JUDGE else "none configured")
print("faithfulness    :", scores.faithfulness)
print("answer relevancy:", scores.answer_relevancy)
if scores.skipped:
    print("skipped:", scores.skipped)

> **Cost guard.** With `USE_PAID_JUDGE = False` and no key set, RAGAS may simply
> report a missing key — that is fine, **skip this cell**. Cell 5 handles a skipped
> Cell 4 without failing. Citation verification in Cell 3 already measures grounding
> deterministically and for free, and is the claim to lead with.
>
> **Why a later paid pass is worth ~a few dollars.** Judging Llama's answers with
> Llama is self-evaluation — a reviewer can fairly object that the model marked its
> own homework. Re-running with `USE_PAID_JUDGE = True` (~22 queries, ~100 short
> calls) gives *"generated locally, judged independently"*, and reporting both
> judges side by side is stronger than either alone.

## Cell 5 — save the results

In [ ]:
import json, pathlib, datetime

# Cell 4 is optional — if it was skipped, carry on and record that RAGAS was
# not computed rather than failing the whole run at the last step.
try:
    scores
except NameError:
    from src.eval.ragas_eval import RAGASScores
    scores = RAGASScores(skipped={"_all": "Cell 4 skipped - no judge configured"})

out = {
    "generated_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "status": "PRELIMINARY - silver query set, book-level labels, "
              "pending the expert gold-set",
    "k": K,
    "n_answerable": full_eval["n_answerable"],
    "n_negative": full_eval["n_negative"],
    "retrieval": [r.as_row(K) for r in full_eval["retrieval"]],
    "generation": {
        "grounded_answer_rate": g.grounded_answer_rate,
        "valid_citation_rate": g.valid_citation_rate,
        "honest_refusal_rate": g.honest_refusal_rate,
        "over_refusal_rate": g.over_refusal_rate,
        "per_query": g.per_query,
    },
    "ungrounded_control": {k: v for k, v in u.items() if k != "answers"},
    "ragas": scores.as_row(),
    "ragas_skipped": scores.skipped,
    "ragas_judge": ("gpt-4o-mini (independent)"
                    if globals().get("USE_PAID_JUDGE") else "none / local"),
}
# Name the file after the judge so a later independent-judge pass ADDS a second
# data point instead of overwriting this one.
suffix = "paidjudge" if globals().get("USE_PAID_JUDGE") else "free"
p = pathlib.Path("results"); p.mkdir(exist_ok=True)
target = p / f"phase11_results_{suffix}.json"
target.write_text(json.dumps(out, indent=2), encoding="utf-8")
print(json.dumps(out["retrieval"], indent=2))
print(f"\nSaved -> {target}")
print("Download it and share with Claude Code to write up the results.")